In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE Gold_FleetMetrics 
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
AS
SELECT
    VehicleID,
    TO_DATE(Timestamp) AS event_date,
    HOUR(Timestamp) AS event_hour,
    
    AVG(Speed) AS AvgSpeed,
    MAX(Speed) AS MaxSpeed,
    MIN(Speed) AS MinSpeed,
    SUM(CASE WHEN OverSpeedAlert = 'true' THEN 1 ELSE 0 END) AS OverSpeedEvents,
    
    AVG(FuelLevel) AS AvgFuelLevel,
    SUM(CASE WHEN FuelStatus IN ('Low', 'Critical') THEN 1 ELSE 0 END) AS LowFuelEvents,
    
    AVG(Odometer) AS AvgOdometer,
    MAX(Odometer) - MIN(Odometer) AS DistanceTravelled,
    
    AVG(RiskScore) AS AvgRiskScore,
    
    SUM(CASE WHEN TireStatus = 'Alert' THEN 1 ELSE 0 END) AS TireAlerts,
    SUM(CASE WHEN EngineStatus IN ('High','Low') THEN 1 ELSE 0 END) AS EngineAlerts,
    SUM(CASE WHEN GeoFenceStatus = FALSE THEN 1 ELSE 0 END) AS GeoFenceAlerts,
    
    AVG(BatteryVoltage) AS AvgBatteryVoltage,
    SUM(CASE WHEN BatteryVoltage < 11.5 THEN 1 ELSE 0 END) AS LowBatteryEvents,
    
    SUM(CASE WHEN DoorStatus = TRUE THEN 1 ELSE 0 END) AS DoorOpenEvents,
    
    SUM(CASE WHEN AuxEquipment1Status = TRUE THEN 1 ELSE 0 END) AS Aux1Usage,
    SUM(CASE WHEN AuxEquipment2Status = TRUE THEN 1 ELSE 0 END) AS Aux2Usage,
    SUM(CASE WHEN AuxEquipment3Status = TRUE THEN 1 ELSE 0 END) AS Aux3Usage,
    SUM(CASE WHEN AuxEquipment4Status = TRUE THEN 1 ELSE 0 END) AS Aux4Usage,
    
    AVG(Temperature) AS AvgTemperature,
    AVG(Humidity) AS AvgHumidity,
    
    current_timestamp() AS ingest_timestamp,
    'Silver_FleetSensorEvents' AS source_system

FROM default.Silver_FleetSensorEvents
GROUP BY VehicleID, TO_DATE(Timestamp), HOUR(Timestamp);